In [ ]:
# Supplemental Figure 4 C, D, E, out of patch content

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.pyplot import cm
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib as mpl

from paths import DATA_DIR, fig_dir


In [ ]:
from spyglass.common import Session

In [ ]:
# custom schema
from find_my_data import *
from alison_decoding import ClusterlessAcausalResultsSummary
from fig_helpers import *
# from plot_out_of_patch import *

In [ ]:
SMALL_SIZE = 20
MEDIUM_SIZE = 20
BIGGER_SIZE = 20

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

set_figure_defaults()

In [ ]:
save_fig = False

fig_path = fig_dir('figs26')
if not os.path.exists(fig_path):
    os.makedirs(fig_path)

custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05]))


### set up and load data

In [ ]:
# Params
position_info_param_name='default_decoding'
remove_hpd_timepoints = True
hpd_percent = 50
hpd_threshold = 50
require_nonlocal_by_segment = False
remove_low_speed_timepoints = True
head_speed_threshold = 10

In [ ]:
out_path = f'{DATA_DIR}/big_df_pkls/'
# today_now = datetime.now().strftime("%Y%m%d") 
today_now = '20240212'
subject_ids = ['senor', 'chimi', 'j16', 'wilbur', 'peanut']

In [ ]:
big_dfs = {}
for subject_id in subject_ids:
    try:
        big_dfs[subject_id] = pd.read_pickle(out_path+subject_id.lower()+'_big_df_RL_deltaq_stable'+today_now+'.pkl')
    except Exception as e:
        print('exception',e)

In [ ]:
stable_nwbs = {}
clusterless_nwbs = {}
stable_clusterless_nwbs = {}
for subject_id in subject_ids:
    stable_nwbs[subject_id] = list( (Session & {'session_description LIKE "Spatial bandit task (regular)"'}
                                             & {"subject_id": subject_id}).fetch('nwb_file_name') )
    clusterless_nwbs[subject_id] = list(np.unique((ClusterlessAcausalResultsSummary()
                                                   & spatial_bandit_query_by_rat(rat_list=[subject_id])).fetch('nwb_file_name')))
    if subject_id == 'j16':
        stable_nwbs['j16'].remove('mediumnwb20230802_.nwb')
    if subject_id == 'chimi':
        stable_nwbs['chimi'].remove('chimi20200216_new_.nwb')
    if subject_id == 'senor':
        stable_nwbs['senor'].remove('senor20201030_.nwb')

    stable_clusterless_nwbs[subject_id] = [nwb for nwb in clusterless_nwbs[subject_id] if nwb in stable_nwbs[subject_id]]

print(stable_clusterless_nwbs)

In [ ]:
is_mapped_seg_a_leaf_map = {0:False, 1:True, 2:True, 3:False, 4:True, 5:True, 6:False, 7:True, 8:True}
segs_to_patch_map = {0:1, 1:1, 2:1, 3:2, 4:2, 5:2, 6:3, 7:3, 8:3}

# get to stable data only
all_rat_big_dfs_stable = {}
for subject_id in subject_ids:
    df = big_dfs[subject_id]
    df_stable = df[df['nwb_file_name'].isin(stable_clusterless_nwbs[subject_id])]
    df_stable['is_actual_seg_mapped_a_leaf'] = df_stable[['actual_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    df_stable['is_mental_seg_mapped_a_leaf'] = df_stable[['mental_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    df_stable['mental_patch_mapped'] = df_stable[['mental_segment_mapped']].applymap(segs_to_patch_map.get)
    all_rat_big_dfs_stable[subject_id] = df_stable

for subject_id in subject_ids:
    df = all_rat_big_dfs_stable[subject_id]
    p_rew_cols = [f"p_rew_leaf{i}" for i in [1,2,3,4,5,6]]
    all_rat_big_dfs_stable[subject_id] = all_rat_big_dfs_stable[subject_id][~all_rat_big_dfs_stable[subject_id][p_rew_cols].eq(all_rat_big_dfs_stable[subject_id]['p_rew_leaf1'], axis=0).all(axis=1)]

In [ ]:
all_rat_big_dfs_patch = {}
for subject_id in subject_ids:
    for is_first_seg_of_trial in [True,False]:
        df = all_rat_big_dfs_stable[subject_id]
        df_seg = df[df['is_first_seg_of_trial']==is_first_seg_of_trial].reset_index()
        bout_first_rows = df_seg.groupby(['nwb_file_name','epoch_number','try_bout_idx']).first().reset_index()
        bout_first_rows['next_patch'] = bout_first_rows.groupby(by=['nwb_file_name','epoch_number'])['stemchoice'].shift(-1) # do the shifting at the level of the bout, now that only one row per bout
        bout_first_rows['prior_patch'] = bout_first_rows.groupby(by=['nwb_file_name','epoch_number'])['stemchoice'].shift(1)
        # give same day ep bout identifier to original and tmp dfs
        df_seg['group_id'] = df_seg['nwb_file_name']+'_'+df_seg['epoch_number'].astype(str) + '_' + df_seg['try_bout_idx'].astype(str)
        bout_first_rows['group_id'] = bout_first_rows['nwb_file_name']+'_'+bout_first_rows['epoch_number'].astype(str) + '_' + bout_first_rows['try_bout_idx'].astype(str)

        # merge
        if is_first_seg_of_trial==True:
            df_first = df_seg.merge(bout_first_rows[['group_id', 'next_patch', 'prior_patch']], on=['group_id'], how='left')
            print(f'{subject_id}: First seg len {len(df_first)}.')
        elif is_first_seg_of_trial==False:
            df_last = df_seg.merge(bout_first_rows[['group_id', 'next_patch', 'prior_patch']], on=['group_id'], how='left')
            print(f'{subject_id}: Last seg len {len(df_last)}.')
        
    # combine first final seg
    df_first_last = pd.concat([df_first,df_last])
    df_sorted = df_first_last.sort_values(by='time')
    print(f'{subject_id}: Full concat len {len(df_sorted)}.\n')
    df_sorted['is_mental_patch_next_patch'] = (df_sorted['mental_patch_mapped'] == df_sorted['next_patch'])
    df_sorted['is_mental_patch_prior_patch'] = (df_sorted['mental_patch_mapped'] == df_sorted['prior_patch'])
    df_sorted['is_mental_patch_chosen_patch'] = (df_sorted['mental_patch_mapped'] == df_sorted['stemchoice'])
    all_rat_big_dfs_patch[subject_id] = df_sorted

In [ ]:
## include jump content (this does not isolate jump content, just directly includes in nonlocal by seg)    

hpd_percent = 50 # or 95
hpd_thresh_cm = 50 # 50
use_abs_ahbeh_thresh = False
abs_ahbeh_thresh_cm = 15
quantile = .9

big_dfs_firstlast_nonlocal_incljump_grouped = {}
for subject_id in subject_ids:
    big_df = all_rat_big_dfs_patch[subject_id]

    # limit to first or last, and hpd and ahbeh restrictions for quality control
    big_df_firstlast = big_df[np.logical_and(
                                    np.logical_or(big_df['is_first_seg_of_trial']==True,
                                                  big_df['is_last_seg_of_trial']==True),
                                    big_df[f'spatial_coverage_{hpd_percent}_hpd']<hpd_thresh_cm,
                                    )]
    if use_abs_ahbeh_thresh:
        big_df_firstlast = big_df_firstlast[big_df_firstlast['abs_ahead_behind_distance']>=abs_ahbeh_thresh_cm]

    big_df_firstlast_nonlocal = big_df_firstlast[big_df_firstlast['nonlocal_by_segment']==True]
   
    # now do all the groupings via lambda fxns - to calc things for first/final seg
    # these calcultaions are only during nonlocal by patch times
    big_df_grouped = big_df_firstlast_nonlocal.groupby(
        by=['nwb_file_name', 'epoch_number', 'trial_number_by_epoch', 'stem_switch', 'stem', 'leaf', 'stemchoice', 'reward', 'is_first_seg_of_trial',
            'trials_from_prior_switch', 'trials_from_next_switch',]
        ).apply(
            lambda x_df: pd.Series({
                'prop_outofpatch_in_chosen': len(x_df[x_df['is_mental_patch_chosen_patch'] & x_df['nonlocal_by_patch']])/len(x_df[x_df['nonlocal_by_patch']]) if len(x_df[x_df['nonlocal_by_patch']])>0 else np.nan, # chosen patch is possible to be local or nonlocal by patch
                'prop_outofpatch_in_prior': len(x_df[x_df['is_mental_patch_prior_patch'] & x_df['nonlocal_by_patch']])/len(x_df[x_df['nonlocal_by_patch']]) if len(x_df[x_df['nonlocal_by_patch']])>0 else np.nan,
                'prop_outofpatch_in_next': len(x_df[x_df['is_mental_patch_next_patch'] & x_df['nonlocal_by_patch']])/len(x_df[x_df['nonlocal_by_patch']]) if len(x_df[x_df['nonlocal_by_patch']])>0 else np.nan,
                'prop_outofpatch_in_leaf': len(x_df[x_df['is_mental_seg_mapped_a_leaf'] & x_df['nonlocal_by_patch']])/len(x_df[x_df['nonlocal_by_patch']]) if len(x_df[x_df['nonlocal_by_patch']])>0 else np.nan,
                'len_outofpatch_in_chosen': len(x_df[x_df['is_mental_patch_chosen_patch'] & x_df['nonlocal_by_patch']]),
                'len_outofpatch_in_prior': len(x_df[x_df['is_mental_patch_prior_patch'] & x_df['nonlocal_by_patch']]),
                'len_outofpatch_in_next': len(x_df[x_df['is_mental_patch_next_patch'] & x_df['nonlocal_by_patch']]),
                'len_outofpatch_in_leaf': len(x_df[x_df['is_mental_seg_mapped_a_leaf'] & x_df['nonlocal_by_patch']]),
                'len_outofpatch': len(x_df[x_df['nonlocal_by_patch']]),
                'len_nonlocal_by_seg': len(x_df),
                'prop_outofpatch_vs_nonlocal_by_seg': len(x_df[x_df['nonlocal_by_patch']])/len(x_df),
                })
            ).reset_index()
       
    big_dfs_firstlast_nonlocal_incljump_grouped[subject_id] = big_df_grouped

In [ ]:
concatenated_df = pd.concat(big_dfs_firstlast_nonlocal_incljump_grouped.values(), keys=big_dfs_firstlast_nonlocal_incljump_grouped.keys(), names=['subject_id'])

# Reset the index to make subject_id a column
concatenated_df.reset_index(level=0, inplace=True)
concatenated_df.reset_index(drop=True, inplace=True)

concatenated_df['zeros'] = 0

### functions

In [ ]:

def prop_trials_out_of_patch_vs_any_nonlocal(big_dfs_firstlast_nonlocal_incljump_grouped, subject_ids, custom_colors_by_rat,
                                            figwidth, figheight, fig_path, marker, s, edgecolors, linewidth,
                                            facecolor=None, transparent=True, pad_inches=.5, save_png=False, save_fig=False):

    plt.figure(figsize=(figwidth,figheight))

    for i,subject_id in enumerate(subject_ids):
        rat_df = big_dfs_firstlast_nonlocal_incljump_grouped[subject_id]
        rat_color = next(custom_colors_by_rat)
        num_segs_nonlocalbyseg = len(rat_df)
        num_segs_notna_gt0 = len(rat_df[rat_df.prop_outofpatch_vs_nonlocal_by_seg.notna() & rat_df.prop_outofpatch_vs_nonlocal_by_seg > 0] )
        prop_trial_segs_with_nonlocal_by_patch = num_segs_notna_gt0/num_segs_nonlocalbyseg
        plt.scatter(x = .03*(i+1)-.06, y = prop_trial_segs_with_nonlocal_by_patch, color=rat_color, marker=marker,s=s, label=f'Rat {subject_id[0].upper()}', edgecolors=edgecolors, linewidth=linewidth)
        plt.legend(frameon=False, bbox_to_anchor=(1.1,.8), loc='upper left')
        plt.ylim(0,1)
        plt.xlim(-1,1)
        #plt.gca().axes.get_xaxis().set_visible(False)
        sns.despine(offset = 5)
        plt.ylabel('Proportion of trial segments\nwith out-of-patch nonlocal/\nsegments with any nonlocal')
        plt.xticks([],[])
        if save_fig:
            fig_name = f'allrats_prop_trial_segs_with_outofpatch_out_of_all_segs_with_nonlocalbyseg'
            save_figure(fig_path, fig_name, facecolor, transparent, pad_inches, save_png)        
    plt.show()

def prop_trials_out_of_patch_vs_any_nonlocal_first_or_final(big_dfs_firstlast_nonlocal_incljump_grouped, subject_ids, custom_colors_by_rat,
                                                            is_first_seg_of_trial,
                                            figwidth, figheight, fig_path, marker, s, edgecolors, linewidth,
                                            facecolor=None, transparent=True, pad_inches=.5, save_png=False, save_fig=False):

    plt.figure(figsize=(figwidth,figheight))

    for i,subject_id in enumerate(subject_ids):
        rat_df = big_dfs_firstlast_nonlocal_incljump_grouped[subject_id]
        rat_df = rat_df[rat_df['is_first_seg_of_trial']==is_first_seg_of_trial]
        rat_color = next(custom_colors_by_rat)
        num_segs_nonlocalbyseg = len(rat_df)
        num_segs_notna_gt0 = len(rat_df[rat_df.prop_outofpatch_vs_nonlocal_by_seg.notna() & rat_df.prop_outofpatch_vs_nonlocal_by_seg > 0] )
        prop_trial_segs_with_nonlocal_by_patch = num_segs_notna_gt0/num_segs_nonlocalbyseg
        plt.scatter(x = .03*(i+1)-.06, y = prop_trial_segs_with_nonlocal_by_patch, color=rat_color, marker=marker,s=s, label=f'Rat {subject_id[0].upper()}', edgecolors=edgecolors, linewidth=linewidth)
        plt.legend(frameon=False, bbox_to_anchor=(1.1,.8), loc='upper left')
        plt.ylim(0,1)
        plt.xlim(-1,1)
        plt.title('First seg' if is_first_seg_of_trial else 'Final seg')
        sns.despine(offset = 5)
        plt.xticks([],[])
        plt.ylabel('Proportion of trial segments\nwith out-of-patch nonlocal/\nsegments with any nonlocal')
        if save_fig:
            fig_name = f'allrats_prop_trial_segs_firstseg{is_first_seg_of_trial}_with_outofpatch_out_of_all_segs_with_nonlocalbyseg'
            save_figure(fig_path, fig_name, facecolor, transparent, pad_inches, save_png)        
    plt.show()

def plot_prop_outofpatch_in_a_patch(concatenated_df, which_patch, is_first_seg_of_trial,stem_switch, show_chance, 
                                   figwidth, figheight, fig_path, marker, s, edgecolors, linewidth,
                                    custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05])),ci=95, dodge=True,
                                            facecolor=None, transparent=True, pad_inches=.5, save_png=False, save_fig=False):
    plt.figure(figsize=(figwidth,figheight))
    subject_colors = {subject_id: next(custom_colors_by_rat) for subject_id in subject_ids}
    concatenated_df_filtered = concatenated_df[concatenated_df.is_first_seg_of_trial == is_first_seg_of_trial]
    concatenated_df_filtered = concatenated_df_filtered[concatenated_df_filtered.stem_switch == stem_switch]
    sns.pointplot(data=concatenated_df_filtered, x ='zeros', y=which_patch, ci=ci, hue='subject_id',palette=subject_colors, dodge=dodge,)
    if show_chance:
        if which_patch == 'prop_outofpatch_in_leaf':
            plt.axhline(4/6, zorder=-1, linestyle='--', color='lightgrey',linewidth=1.5,)
        else:
            plt.axhline(.5, zorder=-1, linestyle='--', color='lightgrey',linewidth=1.5,)
    handles, labels = plt.gca().get_legend_handles_labels()
    plt.legend(frameon=False, bbox_to_anchor=(1.1,.8), loc='upper left', handles=handles, labels=[f'Rat {label[0].upper()}' for label in labels])
    plt.ylim(0,1)
    plt.xticks([],[])
    plt.xlabel('')
    if which_patch == 'prop_outofpatch_in_chosen':
        plt.ylabel('Proportion of Out-of-Patch\nin Chosen Patch')
    elif which_patch == 'prop_outofpatch_in_prior':
        plt.ylabel('Proportion of Out-of-Patch\nin Prior Patch')
    elif which_patch == 'prop_outofpatch_in_next':
        plt.ylabel('Proportion of Out-of-Patch\nin Next Patch')
    elif which_patch == 'prop_outofpatch_in_leaf':
        plt.ylabel('Proportion of Out-of-Patch\nin Leaf Segment')
    else:
        print('That patch calculation does not have a name')
    sns.despine(offset=5)
    plt.title(f'Switch: {stem_switch}, First_seg: {is_first_seg_of_trial}', y=1.05)
    if save_fig:
        fig_name = f'allrats_{yvar}_firstseg{is_first_seg_of_trial}_switch{stem_switch}_ci{ci}_showchance{show_chance}_dodge{dodge}'
        save_figure(fig_path, fig_name, facecolor, transparent, pad_inches, save_png)     
    plt.show()


### plot

In [ ]:

marker='o'
s=85
edgecolors='white'
linewidth=.5

plt.figure(figsize=(2,6))

custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05]))

for i,subject_id in enumerate(subject_ids):
    rat_df = big_dfs_firstlast_nonlocal_incljump_grouped[subject_id]
    rat_color = next(custom_colors_by_rat)
    num_segs_nonlocalbyseg = len(rat_df)
    num_segs_notna_gt0 = len(rat_df[rat_df.prop_outofpatch_vs_nonlocal_by_seg.notna() & rat_df.prop_outofpatch_vs_nonlocal_by_seg > 0] )
    prop_trial_segs_with_nonlocal_by_patch = num_segs_notna_gt0/num_segs_nonlocalbyseg
    plt.scatter(x = .03*(i+1)-.06, y = prop_trial_segs_with_nonlocal_by_patch, color=rat_color, marker=marker,s=s, label=f'Rat {subject_id[0].upper()}', edgecolors=edgecolors, linewidth=linewidth)
    plt.legend(frameon=False, bbox_to_anchor=(1.1,.8), loc='upper left')
    plt.ylim(0,1)
    plt.xlim(-1,1)
    #plt.gca().axes.get_xaxis().set_visible(False)
    sns.despine(offset = 5)
    plt.ylabel('Proportion of trial segments\nwith out-of-patch nonlocal/\nsegments with any nonlocal')
    plt.xticks([],[])
    if save_fig:
        fig_name = f'allrats_prop_trial_segs_with_outofpatch_out_of_all_segs_with_nonlocalbyseg'
        plt.savefig(f'{fig_path}{fig_name}.pdf', format='pdf', bbox_inches="tight", pad_inches=.5)        
plt.show()

In [ ]:
marker='o'
s=85
edgecolors='white'
linewidth=.5
figwidth=2.5
figheight = figwidth*(1/GOLDEN_RATIO)
custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05]))

prop_trials_out_of_patch_vs_any_nonlocal(big_dfs_firstlast_nonlocal_incljump_grouped, subject_ids, custom_colors_by_rat,
                                            figwidth, figheight, fig_path, marker, s, edgecolors, linewidth, save_fig=save_fig)

In [ ]:
marker='o'
s=85
edgecolors='white'
linewidth=.5
figwidth=2.5
figheight = figwidth*(1/GOLDEN_RATIO)
custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05]))

is_first_seg_of_trial = True
prop_trials_out_of_patch_vs_any_nonlocal_first_or_final(big_dfs_firstlast_nonlocal_incljump_grouped, subject_ids, custom_colors_by_rat,is_first_seg_of_trial,
                                            figwidth, figheight, fig_path, marker, s, edgecolors, linewidth, save_fig=save_fig)

custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05]))
is_first_seg_of_trial = False
prop_trials_out_of_patch_vs_any_nonlocal_first_or_final(big_dfs_firstlast_nonlocal_incljump_grouped, subject_ids, custom_colors_by_rat,is_first_seg_of_trial,
                                            figwidth, figheight, fig_path, marker, s, edgecolors, linewidth, save_fig=save_fig)


In [ ]:
# iterate through all of them, though a couple of them are not relevant

ci=95
dodge= True
show_chance = True

yvars = ['prop_outofpatch_in_chosen','prop_outofpatch_in_prior','prop_outofpatch_in_next','prop_outofpatch_in_leaf']

for is_first_seg_of_trial in [True,False]:
    for stem_switch in [True,False]:
        for yvar in yvars:
            custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05]))
            plot_prop_outofpatch_in_a_patch(concatenated_df=concatenated_df,
                                which_patch=yvar,
                                is_first_seg_of_trial=is_first_seg_of_trial,
                                stem_switch=stem_switch,
                                show_chance=show_chance,
                                            figwidth=figwidth,figheight=figheight,fig_path=fig_path,marker=marker,s=s,edgecolors=edgecolors,linewidth=linewidth,
                                ci=ci,
                                dodge=dodge,
                                save_fig=save_fig,
                                custom_colors_by_rat=custom_colors_by_rat,
                               )